In [1]:
import pandas as pd

mavi_sales = pd.read_excel("mavi.xlsx",sheet_name="Sales")
mavi_products = pd.read_excel("mavi.xlsx",sheet_name="Products")
mavi_sales.info() # No N/A values. 
# Time column is in integer type. We need to convert it to datetime type.
mavi_sales["Time"] = pd.to_datetime(mavi_sales["Time"], format="%H%M").dt.time

<class 'pandas.DataFrame'>
RangeIndex: 34070 entries, 0 to 34069
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   DocID            34070 non-null  int64         
 1   StoreCode        34070 non-null  int64         
 2   ProductItemCode  34070 non-null  str           
 3   ProductCode      34070 non-null  str           
 4   Date             34070 non-null  datetime64[us]
 5   ReturnFlag       34070 non-null  int64         
 6   Time             34070 non-null  int64         
 7   Quantity         34070 non-null  int64         
 8   Amount           34070 non-null  float64       
 9   DiscountAmount   34070 non-null  float64       
 10  ChangeCardFlag   34070 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(6), str(2)
memory usage: 2.9 MB


In [2]:
mavi_sales.describe() # As we can see, There is 34070 different rows but DocID's max value is 40000. This is due to some of the DocID values are missing for unknown reasons.

,DocID,StoreCode,Date,ReturnFlag,Quantity,Amount,DiscountAmount,ChangeCardFlag
count,34070.000000,34070.000000,34070,34070.000000,34070.000000,34070.000000,34070.000000,34070.000000
mean,20346.517112,1717.891048,2024-08-17 00:37:14.176695,0.237335,0.525976,523.348707,24.674475,0.007984
min,1.000000,1501.000000,2024-02-01 00:00:00,0.000000,-2.000000,-2545.440000,-727.263640,0.000000
25%,10636.250000,1611.000000,2024-06-03 00:00:00,0.000000,1.000000,408.120065,0.000000,0.000000
50%,20392.500000,1694.000000,2024-09-03 00:00:00,0.000000,1.000000,909.080000,0.000000,0.000000
75%,30115.750000,1837.000000,2024-11-01 00:00:00,0.000000,1.000000,1090.900000,0.000000,0.000000
max,40000.000000,1941.000000,2025-01-31 00:00:00,1.000000,2.000000,2860.110000,1199.990910,1.000000
std,11486.061444,130.529868,NaN,0.425455,0.852167,819.018384,109.155122,0.088995


In [3]:
mavi_sales.sort_values(by="Time") # Negative values in the Quantity, Amount and DiscountAmount columns represents returns.
# Checking if time values are writen correctly.
mavi_sales[mavi_sales.duplicated()] # No duplicates

,DocID,StoreCode,ProductItemCode,ProductCode,Date,ReturnFlag,Time,Quantity,Amount,DiscountAmount,ChangeCardFlag


In [4]:
mavi_sales_positive = mavi_sales[mavi_sales["Quantity"] > 0] # Outlier check.

Q1 = mavi_sales_positive["Amount"].quantile(0.25)
Q3 = mavi_sales_positive["Amount"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

amount_outliers = mavi_sales_positive[
    (mavi_sales_positive["Amount"] < lower_bound) | (mavi_sales_positive["Amount"] > upper_bound)
]

amount_outliers.sort_values(by="Amount", ascending=False)

# These anomaly rows (Quantity > 0 but Amount < 0) were intentionally kept in the dataset.
# They likely represent specific payment behaviors (might be some sort of gift cards not sure) or system corrections. 
# Dropping them would distort the physical inventory count and the actual net revenue.

,DocID,StoreCode,ProductItemCode,ProductCode,Date,ReturnFlag,Time,Quantity,Amount,DiscountAmount,ChangeCardFlag
21510,25681,1901,M1010627-87211028,M1010627-87211,2024-09-27,0,23:29:00,1,2860.11000,0.00000,0
12361,15261,1901,M1010807-88161036,M1010807-88161,2024-06-25,0,23:00:00,1,2822.91200,0.00000,0
14192,17262,1590,M100328-82232017,M100328-82232,2024-08-28,0,15:55:00,2,2545.43636,0.00000,0
19278,22966,1580,M100980-82211042,M100980-82211,2024-09-11,0,11:44:00,2,2423.44000,0.00000,0
31390,36957,1556,M100980-82211042,M100980-82211,2025-01-15,0,19:37:00,2,2369.68000,0.00000,0
...,...,...,...,...,...,...,...,...,...,...,...
19382,23085,1845,M1010299-83039014,M1010299-83039,2024-10-30,0,17:36:00,1,-70.00000,300.00000,0
15885,19091,1803,M1010593-87063014,M1010593-87063,2024-06-19,0,20:03:00,1,-70.00000,300.00000,0
31401,36969,1516,M101077-30503012,M101077-30503,2025-01-13,0,18:10:00,1,-72.13148,327.27273,0
19342,23039,1670,M1010849-88257056,M1010849-88257,2024-09-05,0,16:11:00,1,-76.36000,327.27273,0


In [5]:
mavi_anomaly = mavi_sales[(mavi_sales["Quantity"] > 0) & (mavi_sales["Amount"] < 0)]
mavi_anomaly # Flags rows where a sale (positive Quantity) has a negative Amount — logically inconsistent, likely a discount calculation error rather than a real transaction.

,DocID,StoreCode,ProductItemCode,ProductCode,Date,ReturnFlag,Time,Quantity,Amount,DiscountAmount,ChangeCardFlag
6489,7963,1922,M100277-85272042,M100277-85272,2024-03-13,0,20:24:00,1,-63.64000,272.72727,0
6523,7997,1659,M1010299-83039028,M1010299-83039,2024-03-13,0,21:16:00,1,-57.27000,245.45455,0
6524,7998,1930,M1010125-84418003,M1010125-84418,2024-03-06,0,13:35:00,1,-42.00000,180.00000,0
6535,8011,1894,M101048-84417020,M101048-84417,2024-03-06,0,19:03:00,1,-15.37767,272.72727,0
10756,13390,1666,M101077-85611012,M101077-85611,2024-06-05,0,14:12:00,1,-63.64000,272.72727,0
10757,13391,1534,M101441-81364006,M101441-81364,2024-06-05,0,16:11:00,1,-63.64000,272.72727,0
10780,13416,1892,M1010299-83039007,M1010299-83039,2024-06-12,0,20:28:00,1,-58.79092,300.00000,0
15818,19015,1699,M101441-82297058,M101441-82297,2024-06-05,0,17:56:00,1,-63.64000,272.72727,0
15885,19091,1803,M1010593-87063014,M1010593-87063,2024-06-19,0,20:03:00,1,-70.00000,300.00000,0
19342,23039,1670,M1010849-88257056,M1010849-88257,2024-09-05,0,16:11:00,1,-76.36000,327.27273,0


In [6]:
mavi_productitemcode_group=mavi_sales.groupby("ProductCode").agg(Unique_ProductItemCode_Count=("ProductItemCode","nunique"),
                                                           Total_Count_of_ProductItemCode=("ProductItemCode","count"))
mavi_productitemcode_group # As we can see, for some of the ProductCode values, there are multiple ProductItemCode values. This shows that ProductItemCode column is used to specify the difference between the products with same ProductCode values.

,Unique_ProductItemCode_Count,Total_Count_of_ProductItemCode
ProductCode,,
M100277-21870,1,4
M100277-30104,3,22
M100277-33555,5,1245
M100277-33687,1,82
M100277-35250,1,1190
...,...,...
M101489-35495,1,17
M101489-80543,1,14
M101489-83776,1,1


In [7]:
mavi_merged = pd.merge(mavi_sales, mavi_products, on="ProductCode", how="left") # Since for both tables, ProductCode column is the common column, we can merge them on this column and expand our dataset. We will use left join since we want to keep all the rows in the mavi_sales table and add the corresponding values from the mavi_products table.
mavi_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 34070 entries, 0 to 34069
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   DocID               34070 non-null  int64         
 1   StoreCode           34070 non-null  int64         
 2   ProductItemCode     34070 non-null  str           
 3   ProductCode         34070 non-null  str           
 4   Date                34070 non-null  datetime64[us]
 5   ReturnFlag          34070 non-null  int64         
 6   Time                34070 non-null  object        
 7   Quantity            34070 non-null  int64         
 8   Amount              34070 non-null  float64       
 9   DiscountAmount      34070 non-null  float64       
 10  ChangeCardFlag      34070 non-null  int64         
 11  Class               34070 non-null  str           
 12  MainCategory        34070 non-null  str           
 13  Category            34070 non-null  str           
 14  S

In [8]:
# Fill the missing values in the English columns using the values from the corresponding Turkish columns as reference.
mavi_merged["SubCategoryClassEN"] = mavi_merged["SubCategoryClassEN"].fillna(mavi_merged["SubCategoryClass"])
mavi_merged["SubCategoryEN"] = mavi_merged["SubCategoryEN"].fillna(mavi_merged["SubCategory"])
mavi_merged["CategoryEN"] = mavi_merged["CategoryEN"].fillna(mavi_merged["Category"])
mavi_merged["MainCategoryEN"] = mavi_merged["MainCategoryEN"].fillna(mavi_merged["MainCategory"])
mavi_merged.info() # As we can see, there is no missing values left in the table.

<class 'pandas.DataFrame'>
RangeIndex: 34070 entries, 0 to 34069
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   DocID               34070 non-null  int64         
 1   StoreCode           34070 non-null  int64         
 2   ProductItemCode     34070 non-null  str           
 3   ProductCode         34070 non-null  str           
 4   Date                34070 non-null  datetime64[us]
 5   ReturnFlag          34070 non-null  int64         
 6   Time                34070 non-null  object        
 7   Quantity            34070 non-null  int64         
 8   Amount              34070 non-null  float64       
 9   DiscountAmount      34070 non-null  float64       
 10  ChangeCardFlag      34070 non-null  int64         
 11  Class               34070 non-null  str           
 12  MainCategory        34070 non-null  str           
 13  Category            34070 non-null  str           
 14  S

In [9]:
mavi_merged_mapping = (
    mavi_merged[mavi_merged["Quantity"] > 0]
    .groupby("ProductCode")
    .agg({
        "ProductItemCode": "count", 
        "Quantity": "sum" ,
        "Amount": "sum",              
        "DiscountAmount": "sum",     
        "SubCategoryClass": "first"                 
    })
    .sort_values(by="ProductItemCode", ascending=False)
    .rename(columns={
        "ProductItemCode": "Total_Sales_Count",
        "Quantity": "Total_Units_Sold",
        "Amount": "Total_Revenue",
        "DiscountAmount": "Total_Discount"
    })
)
mavi_merged_mapping["Units Per Transaction"] = (mavi_merged_mapping["Total_Units_Sold"] / mavi_merged_mapping["Total_Sales_Count"]).round(2)

mavi_merged_mapping.round(2)
# This summary table evaluates the commercial performance of each product model by tracking transaction frequency, total physical units sold, total revenue generated, total discounts applied, and the average basket depth (Units Per Transaction).




,Total_Sales_Count,Total_Units_Sold,Total_Revenue,Total_Discount,SubCategoryClass,Units Per Transaction
ProductCode,,,,,,
M101048-84417,1848,1848,1738597.06,7784.78,Slim Straight,1.0
M1010627-87211,1834,1835,1919877.93,91131.81,Straight,1.0
M101225-80680,1385,1385,1333895.39,44094.14,Flare,1.0
M1010299-83039,1174,1174,1110114.12,12247.45,Flare,1.0
M101441-86391,1061,1061,1149567.53,28356.64,Straight,1.0
...,...,...,...,...,...,...
M101437-83751,1,1,252.76,265.41,Flare,1.0
M101441-84307,1,1,500.00,90.90,Straight,1.0
M101441-87636,1,1,909.08,0.00,Straight,1.0


In [10]:
mavi_subcategory_group = mavi_merged_mapping.groupby("SubCategoryClass").sum()

mavi_subcategory_group = mavi_subcategory_group[["Total_Sales_Count", "Total_Units_Sold", "Total_Revenue", "Total_Discount"]].sort_values(by="Total_Revenue", ascending=False)
mavi_subcategory_group["Units Per Transaction"] = (mavi_subcategory_group["Total_Units_Sold"] / mavi_subcategory_group["Total_Sales_Count"]).round(2)
mavi_subcategory_group["Avg_Revenue_Per_Unit"] = (mavi_subcategory_group["Total_Revenue"] / mavi_subcategory_group["Total_Units_Sold"]).round(2)
mavi_subcategory_group["Discount_Rate_%"] = ((mavi_subcategory_group["Total_Discount"] / (mavi_subcategory_group["Total_Revenue"] + mavi_subcategory_group["Total_Discount"])) * 100).round(2)


mavi_subcategory_group
# This summary table reveals the commercial potential of each category by ranking them by total revenue. It evaluates transaction volume, average basket depth, average revenue per unit, and discount rates to identify profitability and discount dependencies.

,Total_Sales_Count,Total_Units_Sold,Total_Revenue,Total_Discount,Units Per Transaction,Avg_Revenue_Per_Unit,Discount_Rate_%
SubCategoryClass,,,,,,,
Straight,5627,5629,5.817281e+06,228516.70855,1.00,1033.45,3.78
Flare,4618,4620,4.532113e+06,153136.11811,1.00,980.98,3.27
Mom,5075,5083,4.193398e+06,345873.62130,1.00,824.98,7.62
Wide Leg,3242,3242,3.437088e+06,150846.95466,1.00,1060.18,4.20
Super Skinny,2696,2707,2.489495e+06,94010.54548,1.00,919.65,3.64
Slim Straight,2210,2210,2.056916e+06,51317.72726,1.00,930.73,2.43
Baggy,1227,1229,1.333697e+06,77997.76386,1.00,1085.19,5.53
Skinny,1096,1097,1.096404e+06,66748.38184,1.00,999.46,5.74
Colored Denims,145,146,1.335450e+05,23822.65468,1.01,914.69,15.14


In [11]:
mavi_merged_refund = (
    mavi_merged[mavi_merged["Quantity"] < 0]
    .groupby("ProductCode")["Quantity"]
    .sum()
    .abs()
    .rename("Total_Returns")
)
mavi_merged_mapping["Total_Returns"] = mavi_merged_refund
mavi_merged_mapping["Total_Returns"] = mavi_merged_mapping["Total_Returns"].fillna(0)
mavi_merged_mapping["Return_Rate_%"] = ((mavi_merged_mapping["Total_Returns"] / mavi_merged_mapping["Total_Units_Sold"]) * 100).round(2)
mavi_subcategory_group["Total_Returns"] = mavi_merged[mavi_merged["Quantity"]<0].groupby("SubCategoryClass")["Quantity"].sum().abs()
mavi_subcategory_group["Return_Rate_%"] = (mavi_subcategory_group["Total_Returns"] / mavi_subcategory_group["Total_Units_Sold"] * 100).round(2)


# This calculates return rates at two levels: per product and per subcategory.
# At the product level, some rates go over 100% (or infinite) — not an error, just because
# a few returns happened for items sold before this dataset's time window (Feb 2024-Jan 2025).
# It mostly affects low-selling products and doesn't change the bigger picture.
# At the subcategory level, this issue goes away since sales volume is much higher,
# making Return_Rate_% a more reliable way to spot fit-related problems across denim types.


In [12]:
mavi_potential_areas = mavi_subcategory_group[
    ["Avg_Revenue_Per_Unit", "Discount_Rate_%", "Return_Rate_%"]
].sort_values("Discount_Rate_%")

mavi_potential_areas # Sorts subcategories by discount dependency to spot which fits sell on
# organic demand (low discount, low returns) vs. which need heavy discounting.
# High efficiency: Slim Straight, Super Skinny, Straight, Wide Leg — low discount rate, solid revenue.
# Low efficiency: Colored Denims, Boyfriend high discount rate, weak underlying demand.

,Avg_Revenue_Per_Unit,Discount_Rate_%,Return_Rate_%
SubCategoryClass,,,
Slim Straight,930.73,2.43,25.75
Flare,980.98,3.27,32.86
Super Skinny,919.65,3.64,30.11
Straight,1033.45,3.78,27.41
Wide Leg,1060.18,4.20,28.25
Baggy,1085.19,5.53,25.14
Skinny,999.46,5.74,38.10
Mom,824.98,7.62,38.46
Colored Denims,914.69,15.14,24.66


In [13]:
mavi_store_sold = mavi_merged[mavi_merged["Quantity"] > 0].groupby("StoreCode")["Quantity"].sum().rename("Total_Sold")
mavi_store_returned = mavi_merged[mavi_merged["Quantity"] < 0].groupby("StoreCode")["Quantity"].sum().abs().rename("Total_Returned")
mavi_store_revenue = mavi_merged[mavi_merged["Quantity"] > 0].groupby("StoreCode")["Amount"].sum().rename("Total_Revenue")

mavi_store_summary = pd.concat([mavi_store_sold, mavi_store_returned, mavi_store_revenue], axis=1).fillna(0)
mavi_store_summary["Return_Rate_%"] = (mavi_store_summary["Total_Returned"] / mavi_store_summary["Total_Sold"] * 100).round(2)
mavi_store_summary.sort_values("Total_Revenue", ascending=False)
# Summarizes sales, returns, and revenue per store, then adds a return rate and sorts by top-earning stores.

,Total_Sold,Total_Returned,Total_Revenue,Return_Rate_%
StoreCode,,,,
1663,259,88.0,255547.76032,33.98
1578,249,78.0,249184.61979,31.33
1636,244,68.0,238940.12865,27.87
1806,236,93.0,237416.23873,39.41
1527,232,77.0,230378.18896,33.19
...,...,...,...,...
1610,6,3.0,5440.96574,50.00
1705,3,0.0,3430.88990,0.00
1707,6,0.0,3177.51956,0.00


In [14]:
mavi_merged.groupby('ChangeCardFlag').agg({"Amount":"mean","Quantity":"mean"})
# All rows with ChangeCardFlag = 1 value have Quantity = 1 value since every change card use also means a purchase.
# # These purchases also average ~2x higher than regular purchases.

,Amount,Quantity
ChangeCardFlag,,
0,519.753353,0.522161
1,970.097790,1.000000


In [15]:
mavi_subcategory_group["Total_ChangeCard"] = (
    mavi_merged[mavi_merged["ChangeCardFlag"] == 1]
    .groupby("SubCategoryClass")["ChangeCardFlag"]
    .count()
)
mavi_subcategory_group["Total_ChangeCard"] = mavi_subcategory_group["Total_ChangeCard"].fillna(0)
mavi_subcategory_group["Total_ChangeCard_Percentage"] = (
    mavi_subcategory_group["Total_ChangeCard"] / mavi_subcategory_group["Total_Sales_Count"] * 100
).round(2)
mavi_subcategory_group

,Total_Sales_Count,Total_Units_Sold,Total_Revenue,Total_Discount,Units Per Transaction,Avg_Revenue_Per_Unit,Discount_Rate_%,Total_Returns,Return_Rate_%,Total_ChangeCard,Total_ChangeCard_Percentage
SubCategoryClass,,,,,,,,,,,
Straight,5627,5629,5.817281e+06,228516.70855,1.00,1033.45,3.78,1543,27.41,55.0,0.98
Flare,4618,4620,4.532113e+06,153136.11811,1.00,980.98,3.27,1518,32.86,31.0,0.67
Mom,5075,5083,4.193398e+06,345873.62130,1.00,824.98,7.62,1955,38.46,58.0,1.14
Wide Leg,3242,3242,3.437088e+06,150846.95466,1.00,1060.18,4.20,916,28.25,41.0,1.26
Super Skinny,2696,2707,2.489495e+06,94010.54548,1.00,919.65,3.64,815,30.11,46.0,1.71
Slim Straight,2210,2210,2.056916e+06,51317.72726,1.00,930.73,2.43,569,25.75,23.0,1.04
Baggy,1227,1229,1.333697e+06,77997.76386,1.00,1085.19,5.53,309,25.14,6.0,0.49
Skinny,1096,1097,1.096404e+06,66748.38184,1.00,999.46,5.74,418,38.10,11.0,1.00
Colored Denims,145,146,1.335450e+05,23822.65468,1.01,914.69,15.14,36,24.66,1.0,0.69


In [16]:
top_changecard = mavi_merged[mavi_merged["ChangeCardFlag"]==1]["ProductCode"].value_counts().head(10)
top_returns = mavi_merged[mavi_merged["Quantity"]<0].groupby("ProductCode")["Quantity"].sum().abs().sort_values(ascending=False).head(10)

top_changecard_and_returns=pd.concat([top_changecard,top_returns],axis=1).dropna()
top_changecard_and_returns.columns = ["ChangeCard_Count", "Total_Returns"]
top_changecard_and_returns

# Cross-references the top 10 products by change-card usage with the top 10
# most-returned products, keeping only the ones that appear in both lists
# revealing which products drive high returns and high exchanges together.
# This is a list of problem-product candidates: products that customers both
# return heavily and exchange often, likely pointing to mislabeled or
# inaccurate size charts for these specific items.

,ChangeCard_Count,Total_Returns
ProductCode,,
M101048-84417,21.0,333.0
M1010627-87211,21.0,255.0
M100277-33555,15.0,274.0
M101072-34111,14.0,342.0
M101441-86391,13.0,260.0
M1010299-83039,12.0,525.0
M100277-35250,10.0,206.0
M101225-80680,10.0,384.0


In [17]:
mavi_sales_positive = mavi_sales[mavi_sales["Quantity"] > 0]

monthly_sales_revenue = mavi_sales_positive.groupby(mavi_sales_positive["Date"].dt.to_period("M"))["Amount"].sum()

monthly_returns = (
    mavi_sales[mavi_sales["Quantity"] < 0]
    .groupby(mavi_sales["Date"].dt.to_period("M"))["Quantity"]
    .sum()
    .abs()
)

monthly_discount = mavi_sales_positive.groupby(mavi_sales_positive["Date"].dt.to_period("M"))["DiscountAmount"].sum()
monthly_amount = mavi_sales_positive.groupby(mavi_sales_positive["Date"].dt.to_period("M"))["Amount"].sum()
monthly_discount_rate = (monthly_discount / (monthly_discount + monthly_amount) * 100).round(2)

monthly_units_sold = mavi_sales_positive.groupby(mavi_sales_positive["Date"].dt.to_period("M"))["Quantity"].sum()
monthly_return_rate = (monthly_returns / monthly_units_sold * 100).round(2)

monthly_avg_order_value = (monthly_sales_revenue / monthly_units_sold).round(2)

mavi_monthly_trend = pd.concat(
    [monthly_sales_revenue, monthly_returns, monthly_discount_rate, monthly_return_rate, monthly_avg_order_value],
    axis=1
).round(2)
mavi_monthly_trend.columns = ["Monthly_Revenue", "Monthly_Returns", "Discount_Rate_%", "Return_Rate_%", "Avg_Order_Value"]

mavi_monthly_trend

# September: highest revenue + lowest return rate the
# healthiest month overall, strong organic demand with minimal returns.
# May: lowest revenue + highest return rate nearly half
# of what sold that month came back, the weakest month by far.
# July: highest discount rate despite mid-range revenue sales
# that month leaned heavily on markdowns rather than natural demand.
# Avg_Order_Value climbs steadily from 838 TL (Feb) to 1126 TL (Jan)
# customers are spending more per unit by year-end, regardless of volume.

,Monthly_Revenue,Monthly_Returns,Discount_Rate_%,Return_Rate_%,Avg_Order_Value
Date,,,,,
2024-02,1105550.28,408,2.94,30.93,838.17
2024-03,1452691.40,442,2.34,25.70,844.59
2024-04,1602172.47,622,1.62,33.57,864.64
2024-05,1014704.95,539,0.98,47.41,892.44
2024-06,2303992.79,905,3.63,35.71,909.23
2024-07,1435455.69,743,11.31,43.94,848.88
2024-08,2076556.29,597,7.82,26.68,927.86
2024-09,3538368.97,820,1.91,24.22,1045.00
2024-10,3444624.93,1091,6.22,31.82,1004.56


In [18]:
mavi_monthly_category_trend = (
    mavi_merged[mavi_merged["Quantity"] > 0]
    .groupby([mavi_merged["Date"].dt.to_period("M"), "SubCategoryClass"])["Quantity"]
    .sum()
    .unstack()
    .fillna(0)
)

first_quarter_avg = mavi_monthly_category_trend.iloc[0:3].mean()
last_quarter_avg = mavi_monthly_category_trend.iloc[-3:].mean()

mavi_category_growth = pd.DataFrame({
    "Quantity_Change": (last_quarter_avg - first_quarter_avg).round(0)
})
mavi_category_growth["Growth_%"] = (
    mavi_category_growth["Quantity_Change"] / first_quarter_avg * 100
).round(2)
mavi_category_growth = mavi_category_growth.sort_values("Quantity_Change", ascending=False)
mavi_category_growth["Growth_%"] = mavi_category_growth["Growth_%"].fillna(0)

mavi_category_growth

# Calculates real volume growth using Quantity instead of Amount to remove inflation and price hike effects.
# Compares the first and last 3 months' average sales to reveal true shifts in consumer demand (volume).

,Quantity_Change,Growth_%
SubCategoryClass,,
Straight,639.0,1114.53
Flare,388.0,223.85
Wide Leg,168.0,104.78
Baggy,154.0,4620.00
Colored Denims,0.0,0.00
Boyfriend,-15.0,-100.00
Mom,-87.0,-30.03
Super Skinny,-134.0,-39.30
Skinny,-157.0,-85.17


In [19]:
with pd.ExcelWriter("mavi_powerbi_export.xlsx") as writer:
    mavi_merged.to_excel(writer, sheet_name="Main_Data", index=False)
    mavi_subcategory_group.to_excel(writer, sheet_name="Category_Summary", index=True)
    mavi_store_summary.to_excel(writer, sheet_name="Store_Summary", index=True)
    mavi_monthly_trend.to_excel(writer, sheet_name="Monthly_Trend", index=True)
    mavi_category_growth.to_excel(writer, sheet_name="Category_Growth", index=True)
    top_changecard_and_returns.to_excel(writer, sheet_name="Problematic_Products", index=True)
    
# Saves the generated summary tables into a single Excel file as different sheets for Power BI import.